In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss


### Read data

In [3]:
regular_season_results = pd.read_csv('../data/MRegularSeasonDetailedResults.csv')
detailed_tourney_results = pd.read_csv('../data/MNCAATourneyDetailedResults.csv')
rankings = pd.read_csv('../data/MMasseyOrdinals.csv')
seeds = pd.read_csv('../data/MNCAATourneySeeds.csv')

# kp_rankings = pd.read_csv('../data/kenpom_pre_tourney_snapshot.csv')

regular_season_results_w = pd.read_csv('../data/WRegularSeasonDetailedResults.csv')
detailed_tourney_results_w = pd.read_csv('../data/WNCAATourneyDetailedResults.csv')

mteams = pd.read_csv('../data/MTeams.csv')
wteams = pd.read_csv('../data/WTeams.csv')

seeds_w = pd.read_csv('../data/WNCAATourneySeeds.csv')

# M538 = pd.read_csv('../data/M538.csv')
# W538 = pd.read_csv('../data/W538.csv')

seed_round = pd.read_csv("../data/MNCAATourneySeedRoundSlots.csv")
seeds = pd.read_csv("../data/MNCAATourneySeeds.csv")

first_round_odds_data = pd.read_csv('../data/sky_data/first_round_odds_ncaam.csv')
first_round_odds_data_w = pd.read_csv('../data/sky_data/first_round_odds_ncaaw.csv')


In [18]:
torvik_player_data = pd.read_csv("../data/sky_data/torvik_player_data_2008_2024.csv")

In [4]:
aggregated_player_stats = pd.read_csv("../data/sky_data/aggregated_player_stats.csv")

In [5]:
aggregated_player_stats.columns

Index(['Unnamed: 0', 'Kaggle_Team', 'Season', 'top8_Games_max',
       'top8_Games_mean', 'top8_Games_weighted_mean', 'top8_Games_stdev',
       'top8_Games_cv', 'top8_Games_gini', 'top8_Games_median',
       ...
       'top3_STL_median', 'top3_FTR_max', 'top3_FTR_mean',
       'top3_FTR_weighted_mean', 'top3_FTR_stdev', 'top3_FTR_cv',
       'top3_FTR_gini', 'top3_FTR_median', 'TeamName', 'TeamID'],
      dtype='object', length=317)

In [6]:
sub_df = pd.read_csv("SampleSubmission2024.csv")

### Set Up Data

In [7]:
to_predict_mens, to_predict_womens, regular_season_games, regular_season_games_w = preprocess.full_setup(detailed_tourney_results, regular_season_results,
               detailed_tourney_results_w, regular_season_results_w,
               sub_df, mteams)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/development_notebooks/../kaggle_prediction_library/preprocess.py:54: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full = df_winners.append(df_losers)
/Users/skylerdale/workspace/kaggle-ncaam/2025_model/development_notebooks/../kaggle_prediction_library/preprocess.py:54: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full = df_winners.append(df_losers)


### Add Features

In [8]:
to_predict_womens = feature_engineering.TournamentSeed(tourney_seeds=seeds_w).add(to_predict_womens)
to_predict_womens = feature_engineering.Efficiency(games=regular_season_games_w, away_bonus=0).add(to_predict_womens)
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.TeamNames(wteams).add(to_predict_womens)

# to_predict_womens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=W538).add(to_predict_womens)

In [9]:
to_predict_womens = feature_engineering.FirstRoundOdds(first_round_odds_data_w).add(to_predict_womens)

In [10]:
# to remove later
to_predict_womens = to_predict_womens[(to_predict_womens.type != "Prediction")].copy()

In [11]:
#to_predict_womens.to_csv("../development_notebooks/to_predict_women.csv")
to_predict_womens.to_csv("to_predict_women.csv")

In [12]:
to_predict_mens = feature_engineering.TeamNames(mteams).add(to_predict_mens)
to_predict_mens = feature_engineering.FirstRoundOdds(first_round_odds_data).add(to_predict_mens)
to_predict_mens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_mens)
to_predict_mens = feature_engineering.SeasonStats(regular_season_games).add(to_predict_mens)
# to_predict_mens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=M538).add(to_predict_mens)
to_predict_mens = feature_engineering.PreSeasonAPRankings(rankings_df=rankings).add(to_predict_mens)
to_predict_mens = feature_engineering.TournamentSeed(tourney_seeds=seeds).add(to_predict_mens)
to_predict_mens = feature_engineering.Efficiency(games=regular_season_games, away_bonus=0).add(to_predict_mens)
to_predict_mens = feature_engineering.FinalRanking(rankings_df=rankings, system='WLK').add(to_predict_mens) # switched in 2024 because SAG dissapeared
# to_predict_mens = feature_engineering.Kenpom(kp_snapshot=kp_rankings).add(to_predict_mens)
# this one takes 3 minutes to run
# to_predict_mens = feature_engineering.TeamQuality(games=regular_season_games).add(to_predict_mens)


In [13]:
to_predict_mens = feature_engineering.AggregatedPlayerStats(aggregated_player_stats).add(to_predict_mens)


In [14]:
to_predict_mens = to_predict_mens[(to_predict_mens.type != "Prediction") & (to_predict_mens.final_odds.notnull())]

### Split Dataset

In [15]:
first_round_df = to_predict_mens[to_predict_mens.GameRound == 1].copy()
other_rounds_df = to_predict_mens[to_predict_mens.GameRound > 1].copy()

In [16]:
# first_round_df.to_csv("to_predict_mens_first_round.csv")
# other_rounds_df.to_csv("to_predict_mens_other_rounds.csv")
to_predict_mens.to_csv("to_predict_mens.csv")

In [17]:
list(to_predict_mens.columns)

['type',
 'ID',
 'Pred',
 'Season',
 'Team1',
 'Team2',
 'Outcome',
 'Gender',
 'margin',
 't1_TeamName',
 't1_FirstD1Season',
 't1_LastD1Season',
 't2_TeamName',
 't2_FirstD1Season',
 't2_LastD1Season',
 'final_odds',
 'GameRound',
 't1_FGM',
 't1_FGA',
 't1_FGM3',
 't1_FGA3',
 't1_OR',
 't1_Ast',
 't1_TO',
 't1_Stl',
 't1_PF',
 't1_FTA',
 't1_FTM',
 't1_PointDiff',
 't2_FGM',
 't2_FGA',
 't2_FGM3',
 't2_FGA3',
 't2_OR',
 't2_Ast',
 't2_TO',
 't2_Stl',
 't2_PF',
 't2_FTA',
 't2_FTM',
 't2_PointDiff',
 't1_OrdinalRank',
 't2_OrdinalRank',
 't1_Seed',
 't2_Seed',
 'seed_diff',
 't1_adj_oe',
 't1_adj_de',
 't1_adj_margin',
 't2_adj_oe',
 't2_adj_de',
 't2_adj_margin',
 't1_final_rank',
 't2_final_rank',
 't1_top8_Games_max',
 't1_top8_Games_mean',
 't1_top8_Games_weighted_mean',
 't1_top8_Games_stdev',
 't1_top8_Games_cv',
 't1_top8_Games_gini',
 't1_top8_Games_median',
 't1_top8_Min%_max',
 't1_top8_Min%_mean',
 't1_top8_Min%_stdev',
 't1_top8_Min%_cv',
 't1_top8_Min%_gini',
 't1_top8_M